<a href="https://colab.research.google.com/github/Denis2054/Context-Engineering-for-Multi-Agent-Systems/blob/main/langchain/Universal_Context_Engine_LangChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Universal Context Engine — LangChain Edition

Copyright 2025-2026, Denis Rothman. LangChain port of the Universal Context Engine.

**Goal:** the same Universal Context Engine, rebuilt on LangChain and LangGraph — reading the *same* Pinecone index, the *same* namespaces, and using the *same* OpenAI models as the original. What this demonstrates is not that one framework beats another. It is that the architecture is portable: the plan artifact, the dual-RAG split, the four specialists and the glass-box trace all survive a complete change of substrate.

| Setting | Value |
|---|---|
| Pinecone index | `genai-mas-mcp-ch3` |
| Blueprint namespace | `ContextLibrary` (metadata field `blueprint_json`, top&#8209;k 1) |
| Knowledge namespace | `KnowledgeStore` (metadata fields `text`, `source`, top&#8209;k 3) |
| Generation model | `gpt-5.1` |
| Embedding model | `text-embedding-3-small` (1536 dims) |

**Nothing is written to Pinecone.** This notebook only reads the vectors the Chapter 8 and Chapter 9 ingestion notebooks already created. There is no re-embedding and no new index.

---

### ⚠️ Prerequisite — the index must already be populated

Run these two notebooks first, in this order:

1. `Chapter08/Data_Ingestion.ipynb` — legal data.
2. `Chapter09/Data_Ingestion_Marketing.ipynb` — **with `clear_index=False`**, so marketing data is appended rather than replacing the legal data.

Section I below runs `lc_utils.check_index()`, which verifies the index dimension and confirms both namespaces contain vectors before you spend anything on a run. If that check fails, stop and fix the ingestion — every downstream cell will otherwise return plausible-looking emptiness with no error.

---

### What LangChain absorbs, and what it does not

| Hand-built in the book | Provided by the framework | Still hand-built here |
|---|---|---|
| `call_llm_robust()` + `tenacity` | `ChatOpenAI(...).with_retry()` | — |
| `json_mode=True` + `json.loads()` + hotfix patch | `.with_structured_output(Plan)` | the `Plan` / `PlanStep` / `AgentInput` schema itself |
| `get_embedding()` | `OpenAIEmbeddings(...)` | — |
| `query_pinecone()` | `PineconeVectorStore(...).as_retriever()` | the two-namespace, two-`text_key` split |
| 4 agent functions + MCP envelopes | 4 `@tool`-decorated LCEL chains | the four system prompts |
| `AgentRegistry` + hand-written capabilities | tool `args_schema` generation | the `ROLE:`/`INPUTS:` rendering the Planner reads |
| execution loop | LangGraph `StateGraph` | Plan-and-Execute itself; `$$STEP_N_OUTPUT$$` context chaining |
| `count_tokens()` (tiktoken estimate) | `usage_metadata` (real billed tokens) | per-**step** attribution against the plan cursor |
| `ExecutionTrace` | LangSmith run tree | the audit-shaped local trace and its offline dashboard |
| `helper_sanitize_input()` | *nothing* | per-chunk injection screening, in full |
| `helper_moderate_content()` | *nothing in v1* | the moderation call and its fail-safe policy |

Roughly two thirds of this codebase is framework. The remaining third — the right-hand column — is the engine. See **Section VII** for the full account.

The original notebook's **Robust Planner hotfix cell is not reproduced**. It existed because the model sometimes returned the plan in the wrong JSON shape; a Pydantic schema with a `Literal` agent field makes that shape error structurally impossible.

---

### How to run

1. Add your keys to **Colab Secrets** (key icon in the left sidebar):
   `API_KEY` (OpenAI), `PINECONE_API_KEY`, and optionally `LANGSMITH_API_KEY`.
2. **Runtime → Run all**, or run Sections I and II then pick individual Control Decks.
3. No GPU needed. Set the runtime to CPU.


# I. Initialization


## Engine modules

This notebook is **standalone**. The seven cells below write the engine's modules
into the Colab filesystem with `%%writefile` — nothing is downloaded, and there is
no dependency on a repository being reachable at run time.

The standalone `.py` files shipped alongside this notebook are byte-identical to
these cells, so you can either run the notebook as-is or import the modules into
your own project. Editing a cell and re-running it replaces the module on disk;
restart the runtime afterwards so the import cache picks up the change.

| File | Replaces | Contains |
|---|---|---|
| `lc_utils.py` | `utils.py` | installation, Colab Secrets, index diagnostics |
| `lc_helpers.py` | `helpers.py` | model, embeddings, two vector stores, guardrails, token tracking |
| `lc_agents.py` | `agents.py` | Librarian, Researcher, Summarizer, Writer as tools |
| `lc_registry.py` | `registry.py` | the toolkit; capabilities generated from schemas |
| `lc_engine.py` | `engine.py` | Plan schema, planner, LangGraph orchestrator, trace |
| `lc_dashboard.py` | notebook cell | the HTML trace dashboard |
| `lc_boot.py` | — | one call that wires everything together |


In [ ]:
%%writefile lc_utils.py
# lc_utils.py
# =============================================================================
# Universal Context Engine — LangChain Edition
# Installation, credentials, and index diagnostics.
#
# Replaces: utils.py
#
# Difference from the original: the original built an `openai.OpenAI` client and
# a `pinecone.Pinecone` client and passed them around by hand. LangChain reads
# credentials from environment variables and builds its own clients, so all this
# module does is move the Colab Secrets into os.environ.
#
# HONEST NOTE ON THE DEPENDENCY SURFACE
# -------------------------------------
# check_index() below imports the Pinecone SDK directly. It is an administrative
# diagnostic, not part of the engine's runtime path — no LangChain abstraction
# exposes describe_index_stats(). It is declared here rather than hidden.
# =============================================================================

from __future__ import annotations

import os
import subprocess
import sys
from typing import Iterable, Optional, Sequence

# Package versions. LangChain moves fast; pin a major range so a notebook that
# works today still works next month. These pins are mirrored in
# requirements.txt — change both together.
PACKAGES = [
    "langchain>=1.0,<2.0",          # create_agent (Route B)
    "langchain-core>=1.0,<2.0",     # Runnables, prompts, tools, documents
    "langchain-openai>=1.0,<2.0",   # ChatOpenAI, OpenAIEmbeddings
    "langchain-pinecone>=0.2,<1.0", # PineconeVectorStore
    "langgraph>=1.0,<2.0",          # StateGraph orchestration
    "pydantic>=2.0,<3.0",           # the Plan schema
    "openai>=1.0,<3.0",             # moderation endpoint fallback (see lc_helpers)
    "pinecone>=5.0,<8.0",           # check_index() only
    "markdown>=3.4",                # the trace dashboard only
]

# The values the engine expects to find in Pinecone. Mirrored in
# lc_helpers.CONFIG; kept here so the pre-flight check can run before the
# engine is built.
EXPECTED_INDEX = "genai-mas-mcp-ch3"
EXPECTED_DIMENSION = 1536           # text-embedding-3-small
EXPECTED_NAMESPACES = ("ContextLibrary", "KnowledgeStore")


# =============================================================================
# 1. Installation
# =============================================================================

def install_dependencies(extra: Optional[Iterable[str]] = None,
                         quiet: bool = True) -> bool:
    """
    Install every package the LangChain engine needs.

    Returns True on success, False on failure. The failure branch prints pip's
    own stderr rather than swallowing it, because a silent install failure
    surfaces later as an ImportError three cells away from its cause.
    """
    pkgs = list(PACKAGES) + list(extra or [])
    print(f"Installing {len(pkgs)} packages...")
    cmd = [sys.executable, "-m", "pip", "install", *pkgs]
    if quiet:
        cmd.append("--quiet")
    try:
        subprocess.run(cmd, check=True, capture_output=quiet, text=True)
        print("All packages installed successfully.")
        return True
    except subprocess.CalledProcessError as e:
        print(f"Installation FAILED (exit code {e.returncode}).")
        if e.stderr:
            print(e.stderr[-2000:])
        return False


# =============================================================================
# 2. Credentials
# =============================================================================

def initialize_environment(
    openai_secret: str = "API_KEY",
    pinecone_secret: str = "PINECONE_API_KEY",
    langsmith_secret: str = "LANGSMITH_API_KEY",
    langsmith_project: str = "universal-context-engine-langchain",
) -> bool:
    """
    Load API keys from Colab Secrets into environment variables.

    LangChain reads OPENAI_API_KEY and PINECONE_API_KEY implicitly, so nothing
    else has to be passed around. Returns True on success.

    The secret names default to the same ones the original notebook used, so an
    existing Colab setup works unchanged. Outside Colab, the function falls back
    to environment variables that are already set.
    """
    print("Initializing environment...")
    try:
        from google.colab import userdata  # noqa: F401
        get = userdata.get
        in_colab = True
    except Exception:
        def get(key):
            return os.environ.get(key)
        in_colab = False
        print("   - Not running in Colab: falling back to existing env vars.")

    try:
        openai_key = get(openai_secret)
        if not openai_key:
            raise ValueError(f"Secret '{openai_secret}' is empty or missing.")
        os.environ["OPENAI_API_KEY"] = openai_key
        print("   - OPENAI_API_KEY set.")

        pinecone_key = get(pinecone_secret)
        if not pinecone_key:
            raise ValueError(f"Secret '{pinecone_secret}' is empty or missing.")
        os.environ["PINECONE_API_KEY"] = pinecone_key
        print("   - PINECONE_API_KEY set.")

        # LangSmith tracing is optional. If the secret is absent we skip it and
        # the engine falls back to its own callback handler and local dashboard.
        try:
            ls_key = get(langsmith_secret)
        except Exception:
            ls_key = None
        if ls_key:
            os.environ["LANGSMITH_API_KEY"] = ls_key
            os.environ["LANGSMITH_TRACING"] = "true"
            os.environ["LANGSMITH_PROJECT"] = langsmith_project
            print(f"   - LangSmith tracing enabled (project: {langsmith_project}).")
        else:
            os.environ["LANGSMITH_TRACING"] = "false"
            print("   - LangSmith not configured (optional). Local tracing only.")

        print("Environment ready.")
        return True

    except Exception as e:
        print(f"Setup failed: {e}")
        if in_colab:
            print("   Add your keys under the key icon in the Colab sidebar:")
            print(f"     {openai_secret}      = your OpenAI key")
            print(f"     {pinecone_secret}    = your Pinecone key")
            print(f"     {langsmith_secret}   = your LangSmith key (optional)")
        else:
            print("   Export them before launching, e.g.:")
            print(f"     export {openai_secret}=sk-...")
            print(f"     export {pinecone_secret}=pcsk_...")
        return False


# =============================================================================
# 3. Pre-flight index diagnostic
# =============================================================================

def check_index(index_name: str = EXPECTED_INDEX,
                namespaces: Sequence[str] = EXPECTED_NAMESPACES,
                expected_dimension: int = EXPECTED_DIMENSION) -> bool:
    """
    Confirm the index exists, has the expected dimension, and that both
    namespaces contain vectors. Returns True only if all three hold.

    This is the cheapest possible way to catch the two failure modes that
    otherwise waste a full run: an unpopulated namespace (zero documents, no
    error) and a dimension mismatch (irrelevant answers, no error).

    Uses the Pinecone SDK directly because this is an administrative check, not
    part of the engine's runtime path.
    """
    try:
        from pinecone import Pinecone
    except ImportError:
        print("The 'pinecone' package is not installed. Run install_dependencies() first.")
        return False

    try:
        pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
        stats = pc.Index(index_name).describe_index_stats()
    except KeyError:
        print("PINECONE_API_KEY is not set. Run initialize_environment() first.")
        return False
    except Exception as e:
        print(f"Could not inspect index '{index_name}': {e}")
        return False

    dimension = stats.get("dimension")
    total = stats.get("total_vector_count")
    print(f"Index '{index_name}' — dimension {dimension}, total vectors {total}")

    ok = True
    if dimension is not None and dimension != expected_dimension:
        print(f"   [FAIL ] dimension is {dimension}, expected {expected_dimension} "
              f"(text-embedding-3-small). This index was written with a different "
              f"embedding model and cannot be read by this engine.")
        ok = False

    ns_stats = stats.get("namespaces", {}) or {}
    for name in namespaces:
        count = (ns_stats.get(name) or {}).get("vector_count", 0)
        mark = "OK   " if count else "EMPTY"
        print(f"   [{mark}] namespace '{name}': {count} vectors")
        if not count:
            ok = False

    if not ok:
        print("\n   Populate the index before running the engine:")
        print("     1. Chapter08/Data_Ingestion.ipynb            (legal data)")
        print("     2. Chapter09/Data_Ingestion_Marketing.ipynb  with clear_index=False")
        print("   The second must append, not replace, or the legal namespace is lost.")
    return ok


In [ ]:
%%writefile lc_helpers.py
# lc_helpers.py
# =============================================================================
# Universal Context Engine — LangChain Edition
# The primitives: model, embeddings, vector stores, retrievers, guardrails,
# and token accounting.
#
# Replaces: helpers.py
#
#   call_llm_robust()        -> ChatOpenAI(...).with_retry()
#   get_embedding()          -> OpenAIEmbeddings(...)
#   query_pinecone()         -> PineconeVectorStore(...).as_retriever()
#   count_tokens()           -> llm.get_num_tokens() + real usage_metadata
#   helper_sanitize_input()  -> sanitize()   [no LangChain equivalent exists]
#   helper_moderate_content()-> moderate()   [no LangChain equivalent in v1]
#   create_mcp_message()     -> not needed; LangGraph state carries structure
#
# WHAT LANGCHAIN DOES NOT PROVIDE, AND IS THEREFORE HAND-WRITTEN HERE
# -------------------------------------------------------------------
#   * sanitize()      — per-chunk prompt-injection screening with skip-and-
#                       continue semantics. Middleware sees a whole message and
#                       cannot drop chunk 2 while keeping chunks 1 and 3.
#   * moderate()      — langchain-core v1 has no moderation wrapper; the legacy
#                       OpenAIModerationChain moved to langchain-classic. This
#                       calls the OpenAI moderation endpoint directly.
#   * UsageTracker    — LangChain reports token usage per call; attributing it
#                       per *plan step* against an execution cursor is ours.
# =============================================================================

from __future__ import annotations

import logging
import re
import warnings
from typing import Any, Dict, List, Optional, Tuple

from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.runnables import RunnableLambda
from langchain_core.vectorstores import VectorStoreRetriever
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)

# =============================================================================
# 1. Configuration — identical values to the original Control Deck
# =============================================================================

CONFIG = {
    "index_name": "genai-mas-mcp-ch3",
    "generation_model": "gpt-5.1",
    "embedding_model": "text-embedding-3-small",
    "namespace_context": "ContextLibrary",
    "namespace_knowledge": "KnowledgeStore",
    # Retrieval depth. These reproduce the original agents' top_k values.
    "k_blueprint": 1,     # agent_context_librarian used top_k=1
    "k_knowledge": 3,     # agent_researcher used top_k=3
    # Which metadata field holds the readable payload in each namespace.
    "text_key_context": "blueprint_json",
    "text_key_knowledge": "text",
    # Retry policy, matching the original tenacity decorator.
    "llm_retries": 6,
    "request_timeout": 120,
}


# =============================================================================
# 2. The model — replaces call_llm_robust
# =============================================================================

def build_llm(config: Optional[dict] = None, **kwargs):
    """
    Build the chat model with the same retry policy the original used.

    Original:  @retry(wait=wait_random_exponential(min=1, max=60),
                      stop=stop_after_attempt(6))
    LangChain: .with_retry(stop_after_attempt=6, wait_exponential_jitter=True)

    The returned object is a Runnable, so the retry wrapper applies to every
    chain it is piped into, not just to one function.

    No temperature is set, exactly as in the original. Do not add one during a
    comparison run: introducing a parameter the original did not use makes
    output differences impossible to attribute.
    """
    cfg = {**CONFIG, **(config or {})}
    base = ChatOpenAI(
        model=cfg["generation_model"],
        timeout=cfg["request_timeout"],
        **kwargs,
    )
    llm = base.with_retry(
        stop_after_attempt=cfg["llm_retries"],
        wait_exponential_jitter=True,
    )
    logging.info(
        f"LLM ready: {cfg['generation_model']} "
        f"(retry: {cfg['llm_retries']} attempts, timeout: {cfg['request_timeout']}s)."
    )
    return llm


def base_model(llm):
    """
    Unwrap a retry-wrapped Runnable back to the underlying ChatOpenAI.

    .with_retry() returns a RunnableRetry that holds the real model in .bound.
    Some methods -- with_structured_output(), get_num_tokens(), and the OpenAI
    client used for moderation -- live on the model itself, not on the wrapper.

    Note that you cannot simply attach an attribute to either object: both are
    Pydantic models and reject unknown fields. This shim exists because of a
    LangChain limitation, not because of a LangChain feature.
    """
    return getattr(llm, "bound", llm)


# =============================================================================
# 3. Embeddings — replaces get_embedding
# =============================================================================

def build_embeddings(config: Optional[dict] = None):
    """
    IMPORTANT: this MUST be the same model that wrote the vectors into Pinecone.
    text-embedding-3-small produces 1536 dimensions. Using anything else returns
    plausible-looking nonsense with no error message.
    """
    cfg = {**CONFIG, **(config or {})}
    emb = OpenAIEmbeddings(model=cfg["embedding_model"])
    logging.info(f"Embeddings ready: {cfg['embedding_model']}.")
    return emb


# =============================================================================
# 4. Vector stores — replaces query_pinecone
#
# One Pinecone index, TWO namespaces, TWO different text_key values.
# This is the single most important detail in the whole port.
# =============================================================================

def build_stores(embeddings,
                 config: Optional[dict] = None
                 ) -> Tuple[PineconeVectorStore, PineconeVectorStore]:
    """
    Returns (context_store, knowledge_store).

    context_store   -> namespace 'ContextLibrary',  text_key 'blueprint_json'
    knowledge_store -> namespace 'KnowledgeStore',  text_key 'text'

    Both READ an existing index. Nothing is written. Never call
    PineconeVectorStore.from_documents() here — that would ingest new vectors,
    chunked differently and with different metadata keys, silently
    desynchronising the index from the Chapter 8 and 9 ingestion notebooks.
    """
    cfg = {**CONFIG, **(config or {})}

    context_store = PineconeVectorStore(
        index_name=cfg["index_name"],
        embedding=embeddings,
        namespace=cfg["namespace_context"],
        text_key=cfg["text_key_context"],
    )
    knowledge_store = PineconeVectorStore(
        index_name=cfg["index_name"],
        embedding=embeddings,
        namespace=cfg["namespace_knowledge"],
        text_key=cfg["text_key_knowledge"],
    )
    logging.info(
        f"Vector stores ready on index '{cfg['index_name']}': "
        f"'{cfg['namespace_context']}' (text_key={cfg['text_key_context']}) and "
        f"'{cfg['namespace_knowledge']}' (text_key={cfg['text_key_knowledge']})."
    )
    return context_store, knowledge_store


def build_retrievers(context_store,
                     knowledge_store,
                     config: Optional[dict] = None
                     ) -> Tuple[VectorStoreRetriever, VectorStoreRetriever]:
    """Turn the two stores into Runnables that take a query string and return a
    list of Documents."""
    cfg = {**CONFIG, **(config or {})}
    blueprint_retriever = context_store.as_retriever(
        search_kwargs={"k": cfg["k_blueprint"]}
    )
    knowledge_retriever = knowledge_store.as_retriever(
        search_kwargs={"k": cfg["k_knowledge"]}
    )
    logging.info(
        f"Retrievers ready: blueprint k={cfg['k_blueprint']}, "
        f"knowledge k={cfg['k_knowledge']}."
    )
    return blueprint_retriever, knowledge_retriever


# =============================================================================
# 5. Security guardrail — same patterns as helper_sanitize_input
#
# NOT a LangChain component. LangChain has no built-in for per-chunk pattern
# screening, and middleware operates at the wrong granularity.
# =============================================================================

INJECTION_PATTERNS = [
    r"ignore previous instructions",
    r"ignore all prior commands",
    r"you are now in.*mode",
    r"act as",
    r"ignore any legal advice",
    r"print your instructions",
    r"sudo|apt-get|yum|pip install",
]

# Compiled once at import. Behaviour is identical to re.search(..., IGNORECASE)
# on every call; this just avoids recompiling on every retrieved chunk.
_COMPILED_PATTERNS = [(p, re.compile(p, re.IGNORECASE)) for p in INJECTION_PATTERNS]


class SanitizationError(ValueError):
    """Raised when a retrieved chunk matches an injection pattern.

    Subclasses ValueError so that `except ValueError` in the original agent code
    continues to work unchanged.
    """

    def __init__(self, pattern: str):
        self.pattern = pattern
        super().__init__("Input sanitization failed. Potential threat detected.")


def sanitize(text: str) -> str:
    """
    Detect prompt-injection patterns in retrieved text.
    Returns the text if clean; raises SanitizationError if a threat is detected.

    This is applied PER RETRIEVED CHUNK inside the Researcher, exactly as in the
    original. Tainted chunks are skipped and the remaining ones still produce an
    answer. Do not move this into agent middleware: middleware sees the whole
    message, so it cannot drop chunk 2 while keeping chunks 1 and 3.
    """
    for raw, compiled in _COMPILED_PATTERNS:
        if compiled.search(text):
            logging.warning(f"[Sanitizer] Potential threat detected with pattern: '{raw}'")
            raise SanitizationError(raw)
    logging.info("[Sanitizer] Input passed sanitization check.")
    return text


# =============================================================================
# 6. Moderation guardrail
#
# langchain-core v1 has no native moderation wrapper (the legacy
# OpenAIModerationChain moved to langchain-classic). We call the OpenAI
# moderation endpoint directly and expose the check as a RunnableLambda so it
# can still be composed and traced like any other engine component.
#
# Client resolution is tiered so that a change in langchain-openai's internals
# degrades to an explicit SDK client instead of breaking moderation outright.
# =============================================================================

_MODERATION_CLIENT = None


def _openai_client(llm=None):
    """
    Resolve an OpenAI SDK client that exposes `.moderations`.

    Order of preference:
      1. ChatOpenAI.root_client  — the public client attribute on the model.
      2. ChatOpenAI.client._client — the parent of the completions resource.
      3. openai.OpenAI() — constructed from OPENAI_API_KEY in the environment.

    Step 3 is why `openai` is a declared dependency rather than an accidental
    transitive one. Relying only on steps 1-2 couples moderation to
    langchain-openai's private attribute layout, which is not a stable contract.
    """
    global _MODERATION_CLIENT
    if _MODERATION_CLIENT is not None:
        return _MODERATION_CLIENT

    if llm is not None:
        base = base_model(llm)
        candidate = getattr(base, "root_client", None)
        if candidate is not None and hasattr(candidate, "moderations"):
            _MODERATION_CLIENT = candidate
            return _MODERATION_CLIENT

        resource = getattr(base, "client", None)
        parent = getattr(resource, "_client", None)
        if parent is not None and hasattr(parent, "moderations"):
            _MODERATION_CLIENT = parent
            return _MODERATION_CLIENT

    from openai import OpenAI  # declared dependency; see PACKAGES in lc_utils
    _MODERATION_CLIENT = OpenAI()
    logging.info("[Moderation] Using a directly constructed OpenAI client.")
    return _MODERATION_CLIENT


def moderate(text: str, llm=None, model: str = "omni-moderation-latest") -> Dict[str, Any]:
    """
    Return a moderation report.

    {
      "flagged":   bool,   # the endpoint flagged the content
      "available": bool,   # the check actually ran
      "categories": {...},
      "scores":     {...},
      "error":      str | None,
    }

    Fail-safe policy: if the check cannot run, `flagged` is True and `available`
    is False. Callers must distinguish the two — "this content is unsafe" and
    "we could not determine whether this content is safe" are different findings
    and, in a regulated deployment, belong in different audit records.
    """
    logging.info("Moderating content...")
    try:
        client = _openai_client(llm)
        result = client.moderations.create(model=model, input=text).results[0]
        report = {
            "flagged": bool(result.flagged),
            "available": True,
            "categories": dict(result.categories),
            "scores": dict(result.category_scores),
            "error": None,
        }
        if report["flagged"]:
            flagged_names = [k for k, v in report["categories"].items() if v]
            logging.warning(f"Content was FLAGGED by moderation: {flagged_names}")
        else:
            logging.info("Content PASSED moderation.")
        return report
    except Exception as e:
        logging.error(f"Moderation error: {e}")
        return {
            "flagged": True,
            "available": False,
            "categories": {},
            "scores": {},
            "error": str(e),
        }


def moderation_runnable(llm=None) -> RunnableLambda:
    """The moderation check as a first-class LangChain Runnable, so it can be
    piped: moderation_runnable(llm) | ... or invoked with .invoke(text)."""
    return RunnableLambda(lambda text: moderate(text, llm), name="Moderation")


# =============================================================================
# 7. Token accounting — replaces count_tokens
#
# Two numbers are tracked, and they mean different things:
#
#   CTX IN / CTX OUT  - the size of the context handed to a step and the size of
#                       what it produced. This is the original notebook's metric
#                       (tiktoken over the payload) and is what the Summarizer's
#                       "tokens saved" figure is computed from.
#   LLM IN / LLM OUT  - the tokens the provider actually billed for this step,
#                       read from AIMessage.usage_metadata. Exact, not estimated.
#                       A retrieval-only step (Librarian) reports 0 here because
#                       it makes no model call.
# =============================================================================

def count_tokens(text: Any, llm=None) -> int:
    """
    Context-size measurement, via LangChain's own tokenizer helper.

    A brand-new model name may not be in tiktoken's registry yet, in which case
    LangChain falls back to a generic tokenizer and warns. The count is then an
    estimate, exactly as the original helpers.count_tokens() was. The warning is
    silenced because it would fire on every step; the exact numbers come from
    usage_metadata (llm_in / llm_out) instead.
    """
    payload = str(text)
    if llm is not None:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                return base_model(llm).get_num_tokens(payload)
        except Exception:
            pass
    return max(1, len(payload) // 4)


class UsageTracker(BaseCallbackHandler):
    """
    Callback handler that accumulates real provider token usage.

    Attach it once per run via config={"callbacks": [tracker]}. The engine takes
    a snapshot before and after each step to attribute usage to that step.

    LangChain reports usage per model call; attributing it per plan step is the
    part LangChain does not do for you.
    """

    def __init__(self):
        self.reset()

    def reset(self) -> "UsageTracker":
        """Clear all counters. Called once per run so figures are per-run."""
        self.input_tokens = 0
        self.output_tokens = 0
        self.llm_calls = 0
        self.events: List[dict] = []
        return self

    # --- snapshotting -----------------------------------------------------
    def snapshot(self) -> Tuple[int, int, int]:
        return (self.input_tokens, self.output_tokens, self.llm_calls)

    def delta(self, snap: Optional[Tuple[int, int, int]]) -> Dict[str, int]:
        if snap is None:
            return {"llm_in": 0, "llm_out": 0, "llm_calls": 0}
        return {
            "llm_in": self.input_tokens - snap[0],
            "llm_out": self.output_tokens - snap[1],
            "llm_calls": self.llm_calls - snap[2],
        }

    # --- LangChain hooks --------------------------------------------------
    def on_llm_end(self, response, **kwargs):
        self.llm_calls += 1
        got = False
        # Preferred: standardized usage_metadata on each generated message.
        try:
            for gen_list in (response.generations or []):
                for gen in gen_list:
                    msg = getattr(gen, "message", None)
                    usage = getattr(msg, "usage_metadata", None) if msg else None
                    if usage:
                        self.input_tokens += usage.get("input_tokens", 0) or 0
                        self.output_tokens += usage.get("output_tokens", 0) or 0
                        got = True
        except Exception:
            pass
        # Fallback: provider-specific llm_output block.
        if not got:
            try:
                tu = (response.llm_output or {}).get("token_usage", {}) or {}
                self.input_tokens += tu.get("prompt_tokens", 0) or 0
                self.output_tokens += tu.get("completion_tokens", 0) or 0
            except Exception:
                pass

    def on_llm_error(self, error, **kwargs):
        self.events.append({"type": "llm_error", "error": str(error)})

    def on_tool_error(self, error, **kwargs):
        self.events.append({"type": "tool_error", "error": str(error)})

    # --- convenience ------------------------------------------------------
    def summary(self) -> Dict[str, int]:
        """Totals for the most recent run only — reset() is called per run."""
        return {
            "llm_calls": self.llm_calls,
            "input_tokens": self.input_tokens,
            "output_tokens": self.output_tokens,
            "total_tokens": self.input_tokens + self.output_tokens,
        }


logging.info("LangChain helper layer defined.")


In [ ]:
%%writefile lc_agents.py
# lc_agents.py
# =============================================================================
# Universal Context Engine — LangChain Edition
# The four specialists, rebuilt as LangChain tools backed by LCEL chains.
#
# Replaces: agents.py
#
#   agent_context_librarian() -> Librarian   (retriever, no LLM call)
#   agent_researcher()        -> Researcher  (high-fidelity RAG with citations)
#   agent_summarizer()        -> Summarizer  (context reduction)
#   agent_writer()            -> Writer      (blueprint + source -> final text)
#
# Two structural changes, both improvements:
#
#   1. The MCP envelope is gone. Tools take typed arguments and return strings,
#      so agent_writer's twenty lines of defensive unpacking (is it a dict? does
#      it have 'facts'? 'summary'? 'answer_with_sources'?) simply disappear.
#      Note that create_mcp_message() was an internal convention resembling the
#      Model Context Protocol, not an implementation of it. Replacing it with
#      typed tool signatures loses no protocol conformance.
#
#   2. The docstrings ARE the capability description. LangChain generates the
#      JSON schema the Planner sees from the function signature and docstring,
#      so registry.get_capabilities_description() no longer has to be written or
#      maintained by hand, and can never drift out of sync with the code.
#
# All system prompts are copied verbatim from the original agents.py so that
# outputs remain comparable.
# =============================================================================

from __future__ import annotations

import json
import logging

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableConfig
from langchain_core.tools import tool

# =============================================================================
# Prompts (verbatim from the original engine)
#
# NOTE ON CURLY BRACES: inside a ChatPromptTemplate, { and } mark a variable.
# Any literal brace in prompt text must be doubled ({{ and }}). None of these
# prompts contain literal braces, but check any prompt you add.
# =============================================================================

RESEARCH_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are an expert research synthesis AI. Your task is to provide a clear, "
     "factual answer to the user's topic based *only* on the provided source "
     "texts. After the answer, you MUST provide a \"Sources\" section listing "
     "the unique source document names you used."),
    ("human",
     "Topic: {topic}\n\nSources:\n{sources}\n\n--- \n"
     "Synthesize your answer and list the source documents now."),
])

SUMMARY_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are an expert summarization AI. Your task is to reduce the provided "
     "text to its essential points, guided by the user's specific objective. "
     "The summary must be concise, accurate, and directly address the stated goal."),
    ("human",
     "--- OBJECTIVE ---\n{objective}\n\n"
     "--- TEXT TO SUMMARIZE ---\n{text}\n--- END TEXT ---\n\n"
     "Generate the summary now."),
])

WRITER_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are an expert content generation AI. Your task is to generate or "
     "rewrite content based on the provided SOURCE MATERIAL, strictly following "
     "the rules in the SEMANTIC BLUEPRINT. The SOURCE MATERIAL may contain both "
     "a synthesized answer and a list of sources; ensure the final output is a "
     "single, cohesive piece of content."),
    ("human",
     "--- SEMANTIC BLUEPRINT (JSON) ---\n{blueprint}\n\n"
     "--- SOURCE MATERIAL ({label}) ---\n{source}\n\n"
     "Generate the final content now."),
])

DEFAULT_BLUEPRINT = json.dumps({"instruction": "Generate the content neutrally."})

NO_DATA_MESSAGE = "No data found on the topic."
ALL_TAINTED_MESSAGE = (
    "Could not generate a reliable answer as retrieved data was suspect."
)


# =============================================================================
# Factory
# =============================================================================

def build_agents(llm, blueprint_retriever, knowledge_retriever, sanitize):
    """
    Build the four specialist tools.

    Dependencies (model, retrievers, sanitizer) are captured in closures. This
    replaces registry.get_handler()'s if/elif dependency-injection ladder: each
    tool simply closes over what it needs, so there is no dispatch table and no
    chance of injecting the wrong argument set into the wrong agent.
    """

    research_chain = RESEARCH_PROMPT | llm | StrOutputParser()
    summary_chain = SUMMARY_PROMPT | llm | StrOutputParser()
    writer_chain = WRITER_PROMPT | llm | StrOutputParser()

    # -------------------------------------------------------------------------
    # 1. Librarian — retrieves a Semantic Blueprint. No LLM call.
    # -------------------------------------------------------------------------
    @tool
    def Librarian(intent_query: str, config: RunnableConfig = None) -> str:
        """Retrieve the Semantic Blueprint that defines the style and structure
        of the required output. Give it a descriptive phrase of the desired
        style or document type, for example "a formal legal summary" or "a
        persuasive marketing pitch". Returns the blueprint as a JSON string."""
        logging.info("[Librarian] Activated. Analyzing intent...")
        docs = blueprint_retriever.invoke(intent_query, config=config)

        if not docs:
            logging.warning("[Librarian] No specific blueprint found. Returning default.")
            return DEFAULT_BLUEPRINT

        doc = docs[0]
        doc_id = getattr(doc, "id", None) or doc.metadata.get("id", "unknown")

        # text_key='blueprint_json' puts the blueprint into page_content.
        # The metadata fallback covers records stored under a different key; if
        # BOTH are empty the text_key is misconfigured, which is worth saying
        # out loud rather than silently degrading to the default blueprint.
        blueprint = doc.page_content or doc.metadata.get("blueprint_json")
        if not blueprint:
            logging.error(
                "[Librarian] Blueprint '%s' retrieved with EMPTY content. This "
                "almost always means text_key is wrong for the ContextLibrary "
                "namespace (expected 'blueprint_json'). Falling back to default.",
                doc_id,
            )
            return DEFAULT_BLUEPRINT

        logging.info(f"[Librarian] Found blueprint '{doc_id}'.")
        return blueprint

    # -------------------------------------------------------------------------
    # 2. Researcher — high-fidelity RAG with per-chunk sanitization + citations.
    # -------------------------------------------------------------------------
    @tool
    def Researcher(topic_query: str, config: RunnableConfig = None) -> str:
        """Retrieve and synthesize factual information on a topic from the
        knowledge base. Use this whenever the goal requires facts, quotations,
        clauses, or any grounded content. Returns the synthesized answer
        followed by a Sources section naming the documents used."""
        logging.info("[Researcher] Activated. Investigating topic with high fidelity...")
        docs = knowledge_retriever.invoke(topic_query, config=config)

        if not docs:
            logging.warning("[Researcher] No relevant information found.")
            return NO_DATA_MESSAGE

        # Sanitize EACH chunk. A poisoned chunk is dropped; the rest still work.
        clean_texts, sources = [], set()
        skipped, empty = 0, 0
        for doc in docs:
            if not (doc.page_content or "").strip():
                empty += 1
                continue
            try:
                clean_texts.append(sanitize(doc.page_content))
                if "source" in doc.metadata:
                    sources.add(doc.metadata["source"])
            except ValueError as e:
                skipped += 1
                logging.warning(
                    f"[Researcher] A retrieved chunk failed sanitization and was "
                    f"skipped. Reason: {e}"
                )
                continue

        if empty:
            logging.error(
                "[Researcher] %d of %d retrieved chunks had EMPTY page_content. "
                "This almost always means text_key is wrong for the KnowledgeStore "
                "namespace (expected 'text').",
                empty, len(docs),
            )

        if not clean_texts:
            logging.error("[Researcher] No usable chunks survived. Aborting.")
            return ALL_TAINTED_MESSAGE

        logging.info(
            f"[Researcher] {len(clean_texts)} of {len(docs)} chunks usable "
            f"({skipped} sanitized out, {empty} empty). Synthesizing with citations..."
        )
        findings = research_chain.invoke(
            {"topic": topic_query, "sources": "\n\n---\n\n".join(clean_texts)},
            config=config,
        )

        # Append the source list programmatically as well, for robustness: the
        # model is instructed to cite, but the citation must not depend on it.
        cited = "\n".join(f"- {s}" for s in sorted(sources))
        return f"{findings}\n\n**Sources:**\n{cited}" if cited else findings

    # -------------------------------------------------------------------------
    # 3. Summarizer — context reduction before an expensive generation step.
    # -------------------------------------------------------------------------
    @tool
    def Summarizer(text_to_summarize: str, summary_objective: str,
                   config: RunnableConfig = None) -> str:
        """Reduce a long text to a concise summary guided by a specific
        objective, for example "Extract key technical specifications". Use this
        to control token counts before handing material to the Writer."""
        logging.info("[Summarizer] Activated. Reducing context...")
        if not text_to_summarize or not summary_objective:
            raise ValueError(
                "Summarizer requires 'text_to_summarize' and 'summary_objective'."
            )
        return summary_chain.invoke(
            {"objective": summary_objective, "text": text_to_summarize},
            config=config,
        )

    # -------------------------------------------------------------------------
    # 4. Writer — applies a blueprint to source material.
    # -------------------------------------------------------------------------
    @tool
    def Writer(blueprint: str, facts: str = "", previous_content: str = "",
               config: RunnableConfig = None) -> str:
        """Generate or rewrite the final content by applying a Semantic
        Blueprint to source material. Supply 'blueprint' (from the Librarian)
        plus either 'facts' (from the Researcher or Summarizer) or
        'previous_content' (existing text to be rewritten). This is normally
        the last step of a plan."""
        logging.info("[Writer] Activated. Applying blueprint to source material...")

        source = facts or previous_content
        label = "SOURCE MATERIAL" if facts else "PREVIOUS CONTENT (For Rewriting)"

        if not blueprint or not source:
            raise ValueError(
                "Writer requires a blueprint and either 'facts' or 'previous_content'."
            )

        return writer_chain.invoke(
            {"blueprint": blueprint, "source": source, "label": label},
            config=config,
        )

    return [Librarian, Researcher, Summarizer, Writer]


logging.info("LangChain specialist agents defined.")


In [ ]:
%%writefile lc_registry.py
# lc_registry.py
# =============================================================================
# Universal Context Engine — LangChain Edition
# The agent toolkit.
#
# Replaces: registry.py
#
# The original AgentRegistry did three jobs:
#   1. mapped agent names to functions            -> a plain dict of tools
#   2. injected dependencies per agent (if/elif)  -> closures in lc_agents.py
#   3. hand-wrote get_capabilities_description()  -> GENERATED from the tools
#
# Job 3 is the interesting one. In the original, the capability text listing
# every input key was maintained by hand in a 30-line f-string, and had to be
# kept in sync with four function signatures by discipline alone. Here it is
# derived from the tools' own Pydantic schemas, so it cannot drift.
#
# The rendering format below is ours, not LangChain's. LangChain can describe a
# tool to a model (render_text_description, convert_to_openai_tool); it does not
# produce the ROLE:/INPUTS: block this engine's Planner prompt expects.
# =============================================================================

from __future__ import annotations

import logging
from typing import Any, Dict, List, Set


class AgentToolkit:
    """Holds the tools and describes them to the Planner."""

    def __init__(self, tools: List):
        self.tools = list(tools)
        self.registry: Dict[str, Any] = {t.name: t for t in self.tools}
        if len(self.registry) != len(self.tools):
            raise ValueError("Duplicate tool names in the toolkit.")
        logging.info(f"Agent toolkit initialized: {', '.join(self.registry)}")

    # ------------------------------------------------------------------ #
    def get(self, name: str):
        """Look up a tool by name. Raises the same error the original did."""
        tool = self.registry.get(name)
        if tool is None:
            logging.error(f"Agent '{name}' not found in registry.")
            raise ValueError(f"Agent '{name}' not found in registry.")
        return tool

    # Backwards-compatible alias with the original API.
    get_handler = get

    def names(self) -> List[str]:
        return list(self.registry.keys())

    # ------------------------------------------------------------------ #
    @staticmethod
    def _schema(tool) -> Dict[str, Any]:
        """The tool's auto-generated JSON schema, or an empty dict."""
        schema_model = getattr(tool, "args_schema", None)
        if schema_model is None:
            return {}
        try:
            return schema_model.model_json_schema()
        except AttributeError:          # a plain dict schema
            return dict(schema_model)
        except Exception:
            return {}

    def arg_names(self, name: str) -> Set[str]:
        """
        The exact argument names a tool accepts.

        Used by the executor to drop any key the Planner produced that this
        agent does not take, rather than relying on Pydantic's silent
        extra-field behaviour.
        """
        return set((self._schema(self.get(name)).get("properties") or {}).keys())

    # ------------------------------------------------------------------ #
    def get_capabilities_description(self) -> str:
        """
        Build the capability block the Planner reads, straight from each tool's
        auto-generated schema. Nothing here is hand-written.
        """
        lines = [
            "Available Agents and their required inputs.",
            "CRITICAL: You MUST use the exact input key names provided for each agent.",
            "",
        ]
        for i, tool in enumerate(self.tools, start=1):
            schema = self._schema(tool)
            props = schema.get("properties") or {}
            required = set(schema.get("required") or [])

            role = " ".join((tool.description or "").split())
            lines.append(f"{i}. AGENT: {tool.name}")
            lines.append(f"   ROLE: {role}")
            lines.append("   INPUTS:")
            if not props:
                lines.append("     - (none)")
            for key, spec in props.items():
                kind = spec.get("type", "string")
                if "anyOf" in spec:
                    kind = "/".join(
                        o.get("type", "any")
                        for o in spec["anyOf"]
                        if o.get("type") != "null"
                    ) or "string"
                flag = "required" if key in required else "optional"
                desc = spec.get("description", "")
                suffix = f" — {desc}" if desc else ""
                lines.append(f'     - "{key}": ({kind}, {flag}){suffix}')
            lines.append("")
        return "\n".join(lines)

    # ------------------------------------------------------------------ #
    def schema_report(self) -> str:
        """Human-readable dump used by the notebook's inspection cell."""
        out = []
        for tool in self.tools:
            args = ", ".join((self._schema(tool).get("properties") or {}).keys())
            out.append(f"{tool.name:<12} ({args})")
        return "\n".join(out)


def build_toolkit(tools) -> AgentToolkit:
    return AgentToolkit(tools)


In [ ]:
%%writefile lc_engine.py
# lc_engine.py
# =============================================================================
# Universal Context Engine — LangChain Edition
# The Planner, the Executor, and the Trace.
#
# Replaces: engine.py
#
#   planner()               -> a Pydantic schema + with_structured_output()
#   resolve_dependencies()  -> unchanged in spirit, now operating on graph state
#   context_engine() loop   -> a compiled LangGraph StateGraph
#   ExecutionTrace          -> LangChainTrace (same field names, so the existing
#                              HTML dashboard renders it with almost no change)
#
# WHY THE PLANNER GETS SAFER
# --------------------------
# The original asked for json_mode=True, received a string, ran json.loads() on
# it, and indexed plan_data["plan"]. When the model wrapped things differently
# the whole run died with "NoneType object has no attribute 'get'", which is why
# the notebook shipped a planner_robust_patch hotfix cell.
#
# Here the plan is a Pydantic model:
#   * agent names are a Literal, so an invented agent name is impossible;
#   * arguments are a typed object, so keys cannot be misspelled or placed at
#     the wrong nesting level;
#   * LangChain performs schema generation, JSON-mode configuration, parsing and
#     validation in one call.
# The hotfix cell is therefore not needed and is not reproduced.
#
# WHAT LANGCHAIN DOES NOT PROVIDE HERE
# ------------------------------------
#   * Plan-and-Execute itself. LangChain removed it from core; what survives in
#     langchain-experimental is unmaintained. The two-node graph below is built
#     by hand on the Graph API.
#   * Context chaining. resolve_dependencies() interprets $$STEP_N_OUTPUT$$
#     references that the Planner writes at plan time and the Executor resolves
#     at run time. LangGraph carries state; it does not give the model a
#     dataflow language to declare dependencies in. That is this engine's own.
#   * The audit-shaped trace. LangSmith records a run tree; it does not record
#     planned_input vs resolved_context vs tokens_saved per plan step.
# =============================================================================

from __future__ import annotations

import copy
import logging
import operator
import re
import time
from typing import Annotated, Any, Dict, List, Literal, Optional, TypedDict

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableConfig
from langgraph.graph import END, StateGraph
from pydantic import BaseModel, Field

# The canonical specialist names. If you add an agent you must add it BOTH here
# and to the Literal in PlanStep below — build_engine() asserts that this tuple
# matches the toolkit, so a mismatch fails loudly at assembly time instead of
# quietly at plan time.
AGENT_NAMES = ("Librarian", "Researcher", "Summarizer", "Writer")


# =============================================================================
# 1. The plan schema — replaces json_mode + json.loads + the hotfix patch
# =============================================================================

class AgentInput(BaseModel):
    """
    The union of every argument any specialist accepts. Every field is optional;
    the Planner fills only the ones the chosen agent needs.

    Declaring these explicitly (rather than an open dict) is what removes the
    original's failure mode: the model cannot invent a key, cannot misspell one,
    and cannot hoist arguments out of the input object.
    """
    intent_query: Optional[str] = Field(
        None, description="Librarian: a descriptive phrase of the desired style.")
    topic_query: Optional[str] = Field(
        None, description="Researcher: the subject matter to research.")
    text_to_summarize: Optional[str] = Field(
        None, description="Summarizer: the long text, or $$STEP_N_OUTPUT$$.")
    summary_objective: Optional[str] = Field(
        None, description="Summarizer: a clear goal for the summary.")
    blueprint: Optional[str] = Field(
        None, description="Writer: style instructions, usually $$STEP_N_OUTPUT$$ from the Librarian.")
    facts: Optional[str] = Field(
        None, description="Writer: factual material, usually $$STEP_N_OUTPUT$$ from the Researcher.")
    previous_content: Optional[str] = Field(
        None, description="Writer: existing text to rewrite.")


class PlanStep(BaseModel):
    step: int = Field(description="Step number, starting at 1.")
    agent: Literal["Librarian", "Researcher", "Summarizer", "Writer"] = Field(
        description="Which specialist executes this step.")
    input: AgentInput = Field(description="Arguments for that specialist.")


class Plan(BaseModel):
    plan: List[PlanStep] = Field(description="The ordered execution plan.")


PLANNER_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are the strategic core of the Context Engine. Analyze the user's "
     "high-level GOAL and create a step-by-step EXECUTION PLAN.\n\n"
     "AVAILABLE CAPABILITIES\n---\n{capabilities}\n---\nEND CAPABILITIES\n\n"
     "INSTRUCTIONS:\n"
     "1. Number steps from 1, in execution order.\n"
     "2. Use Context Chaining: where a value must come from an earlier step, "
     "write the literal string \"$$STEP_N_OUTPUT$$\" (N is that step's number) "
     "instead of the value.\n"
     "3. Fill only the input fields the chosen agent actually needs; leave the "
     "rest null.\n"
     "4. A typical plan retrieves a blueprint, researches the facts, then writes "
     "the final content. Insert a Summarizer step when the research is likely "
     "to be long."),
    ("human", "{goal}"),
])


def build_planner(llm, capabilities: str):
    """Return a Runnable that turns a goal string into a validated Plan."""
    from lc_helpers import base_model
    structured = base_model(llm).with_structured_output(Plan).with_retry(
        stop_after_attempt=4, wait_exponential_jitter=True
    )
    chain = PLANNER_PROMPT | structured

    def plan_for(goal: str, config=None) -> Plan:
        return chain.invoke({"goal": goal, "capabilities": capabilities}, config=config)

    plan_for.chain = chain          # exposed so the notebook can inspect it
    return plan_for


# =============================================================================
# 2. Context chaining — the $$STEP_N_OUTPUT$$ substitution
#
# Not a LangChain feature. The Planner writes these references; the Executor
# resolves them against the outputs of earlier steps.
# =============================================================================

_REFERENCE = re.compile(r"\$\$([A-Za-z0-9_]+)\$\$")


def resolve_dependencies(input_params: Any, state: Dict[str, Any]) -> Any:
    """
    Replace $$REF$$ placeholders with data produced by earlier steps.

    Two cases, deliberately handled differently:

      * The whole string is a reference ("$$STEP_1_OUTPUT$$") -> the stored
        object is substituted as-is, preserving its type.
      * A reference is embedded in a longer string ("Summarize
        $$STEP_1_OUTPUT$$ briefly") -> the stored value is interpolated as text.

    An unresolvable reference is left untouched rather than replaced with None,
    so the failure is visible in the trace instead of silently becoming an empty
    argument.
    """
    resolved = copy.deepcopy(input_params)

    def resolve(value):
        if isinstance(value, str):
            whole = _REFERENCE.fullmatch(value)
            if whole:
                key = whole.group(1)
                if key in state:
                    return state[key]
                logging.warning(f"Unresolved context reference: $${key}$$")
                return value

            def substitute(match):
                key = match.group(1)
                if key in state:
                    return str(state[key])
                logging.warning(f"Unresolved context reference: $${key}$$")
                return match.group(0)

            return _REFERENCE.sub(substitute, value)
        if isinstance(value, dict):
            return {k: resolve(v) for k, v in value.items()}
        if isinstance(value, list):
            return [resolve(v) for v in value]
        return value

    return resolve(resolved)


# =============================================================================
# 3. The Trace — same shape as the original ExecutionTrace
# =============================================================================

class LangChainTrace:
    """
    Records the run so the HTML dashboard can render it.

    Field names match the original ExecutionTrace exactly (goal, plan, steps,
    status, final_output, duration; and per step: step, agent, planned_input,
    resolved_context, output, tokens_in, tokens_out, tokens_saved) so the
    dashboard needed no rewrite.

    Four fields are new:
      llm_in / llm_out  - the provider's own billed token counts per step;
      status            - per step, "ok" or "error", so a failed run still
                          renders the steps that did complete;
      moderation        - the pre-flight and post-flight reports, kept on the
                          trace so a blocked run leaves an audit record.
    """

    def __init__(self, goal: str):
        self.goal = goal
        self.plan: Optional[list] = None
        self.steps: List[dict] = []
        self.status = "Initialized"
        self.final_output: Any = None
        self.start_time = time.time()
        self.duration = 0.0
        self.usage: Dict[str, int] = {}
        self.moderation: Dict[str, Any] = {}
        logging.info(f"LangChainTrace initialized for goal: '{goal}'")

    def log_plan(self, plan):
        self.plan = plan
        logging.info("Plan has been logged to the trace.")

    def log_step(self, **row):
        row.setdefault("status", "ok")
        self.steps.append(row)
        logging.info(
            f"Step {row.get('step')} ({row.get('agent')}) logged [{row['status']}]. "
            f"[ctx in: {row.get('tokens_in', 0)}, ctx out: {row.get('tokens_out', 0)}, "
            f"llm in: {row.get('llm_in', 0)}, llm out: {row.get('llm_out', 0)}]"
        )

    def log_moderation(self, phase: str, report: Dict[str, Any]):
        """Record a moderation report under 'pre' or 'post'."""
        self.moderation[phase] = report

    def finalize(self, status, final_output=None, usage=None):
        self.status = status
        self.final_output = final_output
        self.duration = time.time() - self.start_time
        self.usage = usage or {}
        logging.info(
            f"Trace finalized with status '{status}'. Duration: {self.duration:.2f}s"
        )


# =============================================================================
# 4. The graph — replaces the plan-then-execute loop in context_engine()
# =============================================================================

class EngineState(TypedDict, total=False):
    goal: str
    plan: List[dict]
    cursor: int
    outputs: Dict[str, Any]
    steps: Annotated[List[dict], operator.add]   # appended, never overwritten
    final: Any
    error: str


def build_engine(llm, toolkit, tracker=None, count_tokens=None):
    """
    Compile the Plan-and-Execute graph.

    Nodes:
      plan     -> one structured LLM call producing a validated Plan
      execute  -> run one step, merge its output into shared state, loop

    Because this is a compiled LangGraph you get, for free: streaming of
    intermediate state, optional checkpointing so a failed run can resume at the
    failing step, and automatic LangSmith traces of every node.
    """
    toolkit_names = tuple(toolkit.names())
    if set(toolkit_names) != set(AGENT_NAMES):
        raise ValueError(
            f"Toolkit exposes {toolkit_names} but the Plan schema allows "
            f"{AGENT_NAMES}. Add the new agent to AGENT_NAMES and to the "
            f"Literal in PlanStep, or the Planner can never select it."
        )

    planner = build_planner(llm, toolkit.get_capabilities_description())
    _count = count_tokens or (lambda text, _llm=None: max(1, len(str(text)) // 4))

    # ---------------------------------------------------------------- plan
    def plan_node(state: EngineState, config: RunnableConfig = None) -> dict:
        logging.info("Planner activated. Analyzing goal and generating execution plan...")
        try:
            result = planner(state["goal"], config=config)
            plan = [
                {
                    "step": s.step,
                    "agent": s.agent,
                    "input": s.input.model_dump(exclude_none=True),
                }
                for s in result.plan
            ]
            if not plan:
                return {"error": "Planner returned an empty plan."}
            logging.info(f"Plan generated: {len(plan)} step(s).")
            return {"plan": plan, "cursor": 0, "outputs": {}}
        except Exception as e:
            logging.error(f"Planner failed to generate a valid plan. Error: {e}")
            return {"error": f"Failed during Planning/Init: {e}"}

    # ------------------------------------------------------------- execute
    def execute_node(state: EngineState, config: RunnableConfig = None) -> dict:
        step = state["plan"][state["cursor"]]
        num, name, planned = step["step"], step["agent"], step["input"]
        logging.info(f"--- Executor: Starting Step {num}: {name} ---")

        resolved: Any = planned
        snap = tracker.snapshot() if tracker else None

        try:
            tool = toolkit.get(name)
            resolved = resolve_dependencies(planned, state.get("outputs", {}))

            # Drop any argument this agent does not accept. The Plan schema is a
            # union of every specialist's arguments, so a Planner that fills one
            # extra field would otherwise depend on Pydantic's silent
            # extra-field behaviour to survive.
            accepted = toolkit.arg_names(name)
            if isinstance(resolved, dict) and accepted:
                dropped = sorted(set(resolved) - accepted)
                if dropped:
                    logging.warning(
                        f"[Executor] Step {num} ({name}): dropping argument(s) "
                        f"{dropped} — not accepted by this agent."
                    )
                    resolved = {k: v for k, v in resolved.items() if k in accepted}

            t_in = _count(resolved, llm)
            output = tool.invoke(resolved, config=config)
            t_out = _count(output, llm)
            delta = tracker.delta(snap) if tracker else {"llm_in": 0, "llm_out": 0}

            key = f"STEP_{num}_OUTPUT"
            is_last = state["cursor"] + 1 >= len(state["plan"])
            logging.info(f"--- Executor: Step {num} completed. ---")

            return {
                "outputs": {**state.get("outputs", {}), key: output},
                "cursor": state["cursor"] + 1,
                "final": output if is_last else state.get("final"),
                "steps": [{
                    "step": num,
                    "agent": name,
                    "status": "ok",
                    "planned_input": planned,
                    "resolved_context": resolved,
                    "output": output,
                    "tokens_in": t_in,
                    "tokens_out": t_out,
                    # The original computed "saved" only for the Summarizer.
                    "tokens_saved": max(0, t_in - t_out) if name == "Summarizer" else 0,
                    "llm_in": delta.get("llm_in", 0),
                    "llm_out": delta.get("llm_out", 0),
                }],
            }

        except Exception as e:
            msg = f"Execution failed at step {num} ({name}): {e}"
            logging.error(f"--- Executor: FATAL ERROR --- {msg}")
            delta = tracker.delta(snap) if tracker else {"llm_in": 0, "llm_out": 0}
            # Record the failed step too, so the dashboard shows where the run
            # stopped and with what resolved input, rather than ending silently.
            return {
                "error": msg,
                "cursor": state["cursor"] + 1,
                "steps": [{
                    "step": num,
                    "agent": name,
                    "status": "error",
                    "planned_input": planned,
                    "resolved_context": resolved,
                    "output": msg,
                    "tokens_in": _count(resolved, llm),
                    "tokens_out": 0,
                    "tokens_saved": 0,
                    "llm_in": delta.get("llm_in", 0),
                    "llm_out": delta.get("llm_out", 0),
                }],
            }

    # ------------------------------------------------------------- routing
    def route(state: EngineState) -> str:
        if state.get("error"):
            return "end"
        if state.get("cursor", 0) < len(state.get("plan") or []):
            return "execute"
        return "end"

    graph = StateGraph(EngineState)
    graph.add_node("plan", plan_node)
    graph.add_node("execute", execute_node)
    graph.set_entry_point("plan")
    graph.add_conditional_edges("plan", route, {"execute": "execute", "end": END})
    graph.add_conditional_edges("execute", route, {"execute": "execute", "end": END})
    return graph.compile()


# =============================================================================
# 5. The public entry point — same contract as the original context_engine()
# =============================================================================

def context_engine(goal: str, engine, tracker=None, recursion_limit: int = 60,
                   trace: Optional[LangChainTrace] = None):
    """
    Run the engine. Returns (final_output, trace).

    Signature note: the original took client, pc, index_name, models and
    namespaces. All of that is now baked into the compiled `engine` object, so
    the runtime call is just the goal.

    A caller may pass an existing `trace` so that pre-flight moderation, which
    happens before the graph runs, is recorded on the same object.
    """
    logging.info(f"--- [Context Engine] Starting New Task --- Goal: {goal}")
    trace = trace or LangChainTrace(goal)

    config: Dict[str, Any] = {"recursion_limit": recursion_limit}
    if tracker is not None:
        config["callbacks"] = [tracker]

    try:
        state = engine.invoke({"goal": goal, "steps": []}, config=config)
    except Exception as e:
        logging.error(f"Engine invocation failed: {e}")
        trace.finalize(f"Failed: {e}", None, tracker.summary() if tracker else {})
        return None, trace

    if state.get("plan"):
        trace.log_plan(state["plan"])
    for row in state.get("steps", []):
        trace.log_step(**row)

    usage = tracker.summary() if tracker else {}

    if state.get("error"):
        trace.finalize(state["error"], None, usage)
        logging.error(f"--- [Context Engine] Task Failed --- {state['error']}")
        return None, trace

    final = state.get("final")
    trace.finalize("Success", final, usage)
    logging.info("--- [Context Engine] Task Complete ---")
    return final, trace


In [ ]:
%%writefile lc_dashboard.py
# lc_dashboard.py
# =============================================================================
# Universal Context Engine — LangChain Edition
# The HTML trace dashboard.
#
# The CSS and layout are carried over from the original notebook unchanged.
# Three additions:
#   * the metrics bar shows the provider's own billed token counts (LLM IN /
#     LLM OUT) from usage_metadata, alongside the context-size pills;
#   * a step that failed renders in the failure colour with its error text,
#     instead of the run ending with no visible cause;
#   * a moderation line appears when a pre- or post-flight check ran.
#
# Expect the token numbers to differ from the original notebook's screenshots.
# They are measuring different things, and these ones are exact.
#
# There is no LangChain equivalent for this file. LangSmith is a hosted run
# viewer; this is a self-contained, offline, embeddable audit artifact.
# =============================================================================

import html
import json

import markdown
from IPython.display import HTML, display

CSS = """
<style>
    .dashboard-container {
        font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
        background-color: #ffffff;
        border: 3px solid #cbd5e0;
        border-radius: 12px;
        padding: 30px;
        max-width: 100%;
        margin-top: 25px;
        color: #1a202c;
    }
    .header-section {
        border-bottom: 3px solid #2d3748;
        padding-bottom: 20px;
        margin-bottom: 25px;
        display: flex;
        justify-content: space-between;
        align-items: center;
    }
    .header-title { margin: 0; font-size: 1.8rem; color: #1a202c; font-weight: 900; }
    .header-goal { margin: 10px 0 0 0; color: #2d3748; font-size: 1.2rem; font-style: italic; font-weight: 600;}
    .header-engine { margin: 6px 0 0 0; color: #2b6cb0; font-size: 0.9rem; font-weight: 900; text-transform: uppercase; letter-spacing: 1px;}

    .status-badge {
        padding: 10px 20px;
        border-radius: 8px;
        font-weight: 900;
        font-size: 1rem;
        color: white;
        text-transform: uppercase;
    }
    .status-success { background-color: #22543d; }
    .status-failure { background-color: #742a2a; }

    .metrics-summary {
        margin-top: 10px;
        font-size: 1.1rem;
        font-weight: 900;
        color: #1a202c;
        background: #edf2f7;
        padding: 5px 12px;
        border-radius: 6px;
    }

    .metrics-bar { display: flex; gap: 12px; margin-top: 12px; flex-wrap: wrap; }
    .metric-pill {
        background-color: #ebf4ff;
        color: #1a365d;
        padding: 6px 14px;
        border-radius: 8px;
        border: 2px solid #2b6cb0;
        font-size: 0.95rem;
        font-weight: 900;
    }
    .metric-saved { background-color: #f0fff4; color: #1c4532; border-color: #2f855a; }
    .metric-llm   { background-color: #faf5ff; color: #44337a; border-color: #6b46c1; }
    .metric-error { background-color: #fff5f5; color: #742a2a; border-color: #742a2a; }

    .step-card {
        background-color: #ffffff;
        border: 2px solid #2d3748;
        border-radius: 12px;
        margin-bottom: 25px;
        overflow: hidden;
        box-shadow: 0 4px 6px rgba(0,0,0,0.05);
    }
    .step-card-error { border-color: #742a2a; }
    summary.step-header {
        padding: 20px;
        background-color: #f8fafc;
        cursor: pointer;
        list-style: none;
        display: flex;
        align-items: center;
        justify-content: space-between;
        border-bottom: 2px solid #e2e8f0;
    }
    .step-card-error summary.step-header { background-color: #fff5f5; }
    .agent-badge {
        background-color: #1a202c;
        color: #ffffff;
        padding: 5px 14px;
        border-radius: 6px;
        font-size: 0.85rem;
        font-weight: 900;
        text-transform: uppercase;
        margin-left: 15px;
    }
    .step-card-error .agent-badge { background-color: #742a2a; }

    .step-content { padding: 25px; background-color: #ffffff; }

    .data-label {
        font-size: 1rem;
        text-transform: uppercase;
        color: #1a202c;
        font-weight: 900;
        margin-bottom: 12px;
        display: block;
        border-left: 4px solid #1a202c;
        padding-left: 10px;
    }

    .rendered-content {
        background-color: #ffffff;
        border: 2px solid #e2e8f0;
        border-left: 8px solid #2b6cb0;
        padding: 20px;
        color: #1a202c !important;
        line-height: 1.7;
        font-size: 1.1rem;
        font-weight: 500;
    }
    .rendered-content h1, .rendered-content h2, .rendered-content h3,
    .rendered-content h4, .rendered-content h5, .rendered-content h6 {
        color: #1a202c !important;
        font-weight: 900 !important;
        margin-top: 1.5em;
        margin-bottom: 0.5em;
    }
    .rendered-content strong, .rendered-content b { font-weight: 900; color: #000000; }
    .rendered-error { border-left-color: #742a2a; color: #742a2a !important; font-weight: 700; }

    .json-box {
        background-color: #1a202c;
        color: #f7fafc;
        padding: 20px;
        border-radius: 10px;
        font-family: "SFMono-Regular", Consolas, monospace;
        font-size: 0.95rem;
        overflow-x: auto;
        white-space: pre-wrap;
    }

    .final-output-card {
        border: 5px solid #22543d;
        background-color: #f0fff4;
        border-radius: 12px;
        padding: 30px;
        margin-top: 50px;
        color: #1a202c;
    }
</style>
"""

# Keys that may hold the readable payload when a step returns a dict rather
# than a string. Retained from the original for backward compatibility with
# traces produced by the pre-port engine.
_TEXT_KEYS = ("summary", "answer_with_sources", "answer", "output", "content")


def _md(text):
    """Render text as markdown. Empty values become an explicit placeholder."""
    return markdown.markdown(str(text)) if text else "No content recorded."


def _moderation_line(trace):
    """One summary line covering whichever moderation checks actually ran."""
    reports = getattr(trace, "moderation", {}) or {}
    if not reports:
        return ""
    parts = []
    for phase in ("pre", "post"):
        report = reports.get(phase)
        if not report:
            continue
        if not report.get("available", True):
            verdict = "UNAVAILABLE"
        elif report.get("flagged"):
            verdict = "FLAGGED"
        else:
            verdict = "PASS"
        parts.append(f"{phase}-flight {verdict}")
    if not parts:
        return ""
    return f'<div class="metrics-summary">🛡️ MODERATION: {html.escape(" · ".join(parts))}</div>'


def render_trace_dashboard(trace, engine_label="LangGraph Plan-and-Execute"):
    """Render a LangChainTrace as the familiar Context Engine dashboard."""
    status_text = str(getattr(trace, "status", "Unknown"))
    status_class = "status-success" if status_text == "Success" else "status-failure"
    # A failure status carries the full exception text; keep the badge readable
    # and leave the detail to the step card that recorded it.
    badge_text = status_text if len(status_text) <= 40 else status_text[:37] + "..."

    usage = getattr(trace, "usage", {}) or {}
    usage_line = ""
    if usage:
        usage_line = (
            f'<div class="metrics-summary">🧠 BILLED: '
            f'{usage.get("input_tokens", 0)} in / {usage.get("output_tokens", 0)} out '
            f'({usage.get("llm_calls", 0)} calls)</div>'
        )

    out = [CSS, f"""
    <div class="dashboard-container">
        <div class="header-section">
            <div>
                <h1 class="header-title">Context Engine Trace</h1>
                <p class="header-goal">"{html.escape(str(trace.goal))}"</p>
                <p class="header-engine">LangChain Edition &middot; {html.escape(engine_label)}</p>
            </div>
            <div style="text-align: right;">
                <span class="status-badge {status_class}" title="{html.escape(status_text)}">{html.escape(badge_text)}</span>
                <div class="metrics-summary">⏱️ TIME: {getattr(trace, 'duration', 0.0):.2f}s</div>
                {usage_line}
                {_moderation_line(trace)}
            </div>
        </div>
        <div class="steps-container">
            <h2 style="color:#1a202c; margin-bottom:25px; font-size:1.4rem; font-weight:900; text-transform:uppercase;">Execution Workflow</h2>
    """]

    # ---- The plan, shown before the steps -----------------------------------
    if getattr(trace, "plan", None):
        out.append(f"""
            <details class="step-card">
                <summary class="step-header">
                    <div><span style="font-weight:900; font-size:1.3rem; color:#1a202c;">EXECUTION PLAN</span>
                    <span class="agent-badge">Planner</span></div>
                    <span style="font-weight:900; color:#ffffff; background:#1a202c; padding:6px 14px; border-radius:6px; font-size:0.8rem;">OPEN PLAN</span>
                </summary>
                <div class="step-content">
                    <div class="json-box">{html.escape(json.dumps(trace.plan, indent=2, default=str))}</div>
                </div>
            </details>
        """)

    # ---- Each executed step -------------------------------------------------
    for step in getattr(trace, "steps", []):
        failed = step.get("status") == "error"

        try:
            resolved_ctx = json.dumps(step.get("resolved_context"), indent=2, default=str)
        except Exception:
            resolved_ctx = str(step.get("resolved_context", "N/A"))

        output_raw = step.get("output", "N/A")
        if failed:
            rendered = (
                f'<div class="rendered-content rendered-error">'
                f'{html.escape(str(output_raw))}</div>'
            )
        elif isinstance(output_raw, dict):
            for key in _TEXT_KEYS:
                if isinstance(output_raw.get(key), str):
                    rendered = f'<div class="rendered-content">{_md(output_raw[key])}</div>'
                    break
            else:
                rendered = (
                    f'<div class="json-box">'
                    f'{html.escape(json.dumps(output_raw, indent=2, default=str))}</div>'
                )
        else:
            rendered = f'<div class="rendered-content">{_md(output_raw)}</div>'

        pills = [
            f'<span class="metric-pill">📥 CTX IN: {step.get("tokens_in", "??")}</span>',
            f'<span class="metric-pill">📤 CTX OUT: {step.get("tokens_out", "??")}</span>',
        ]
        if (step.get("tokens_saved") or 0) > 0:
            pills.append(
                f'<span class="metric-pill metric-saved">📉 SAVED: {step["tokens_saved"]}</span>'
            )
        if step.get("llm_in") or step.get("llm_out"):
            pills.append(
                f'<span class="metric-pill metric-llm">🧠 LLM: '
                f'{step.get("llm_in", 0)} in / {step.get("llm_out", 0)} out</span>'
            )
        else:
            pills.append('<span class="metric-pill metric-llm">🧠 LLM: retrieval only</span>')
        if failed:
            pills.append('<span class="metric-pill metric-error">⛔ FAILED</span>')

        out.append(f"""
            <details class="step-card{' step-card-error' if failed else ''}" open>
                <summary class="step-header">
                    <div style="display:flex; flex-direction:column; align-items:flex-start;">
                        <div>
                            <span style="font-weight:900; font-size:1.3rem; color:#1a202c;">STEP {step.get('step')}</span>
                            <span class="agent-badge">{html.escape(str(step.get('agent')))}</span>
                        </div>
                        <div class="metrics-bar">{''.join(pills)}</div>
                    </div>
                    <span style="font-weight:900; color:#ffffff; background:#1a202c; padding:6px 14px; border-radius:6px; font-size:0.8rem;">OPEN LOGS</span>
                </summary>
                <div class="step-content">
                    <div style="margin-bottom:30px;">
                        <span class="data-label">Input Context (State)</span>
                        <details><summary style="font-size:0.9rem; font-weight:900; color:#2b6cb0; cursor:pointer; margin-bottom:10px;">▶ View Resolved Source Data</summary>
                        <div class="json-box">{html.escape(resolved_ctx)}</div></details>
                    </div>
                    <div>
                        <span class="data-label">Agent Output</span>
                        {rendered}
                    </div>
                </div>
            </details>
        """)

    # ---- Final result -------------------------------------------------------
    if getattr(trace, "final_output", None):
        final_content = trace.final_output
        if isinstance(final_content, dict):
            final_content = final_content.get(
                "summary", final_content.get("content", str(final_content))
            )
        out.append(f"""
        <div class="final-output-card">
            <div style="color:#1c4532; font-size:1.5rem; font-weight:1000; margin-bottom:20px; text-transform:uppercase; letter-spacing:2px; border-bottom:3px solid #22543d; padding-bottom:12px;">Final Orchestration Result</div>
            <div style="font-size:1.25rem; font-weight:700; line-height:1.8;">{markdown.markdown(str(final_content))}</div>
        </div>
        """)

    out.append("</div></div>")
    display(HTML("".join(out)))


In [ ]:
%%writefile lc_boot.py
# lc_boot.py
# =============================================================================
# Universal Context Engine — LangChain Edition
# Assembly. One call wires every component together.
#
# This file has no counterpart in the original project: the original assembled
# itself inside the notebook by passing `client`, `pc` and a config dict into
# every function. Here the wiring happens once and produces a single object.
# =============================================================================

from __future__ import annotations

import logging
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple

import lc_agents
import lc_engine
import lc_helpers
import lc_registry


@dataclass
class ContextEngine:
    """Everything the engine needs, assembled and ready to run."""
    config: Dict[str, Any]
    llm: Any
    base_llm: Any
    embeddings: Any
    context_store: Any
    knowledge_store: Any
    blueprint_retriever: Any
    knowledge_retriever: Any
    toolkit: Any
    graph: Any
    tracker: Any
    tools: List = field(default_factory=list)
    _planner: Any = None

    # ------------------------------------------------------------------ #
    def run(self, goal: str, moderation_active: bool = False
            ) -> Tuple[Optional[str], lc_engine.LangChainTrace]:
        """
        Execute a goal. Returns (result, trace).

        Mirrors the original notebook's execute_and_display flow: pre-flight
        moderation on the goal, the engine, then post-flight moderation on the
        output. Rendering is left to the caller.

        CONTRACT NOTE: a trace is ALWAYS returned, including when the goal is
        blocked before the engine runs. A blocked goal is an audit event, and an
        audit trail with a hole in it where the refusals should be is not an
        audit trail. Callers should check `trace.status`, not `trace is None`.
        """
        trace = lc_engine.LangChainTrace(goal)

        # A fresh tracker per run so token counts are per-run, not cumulative.
        self.tracker.reset()

        if moderation_active:
            print("--- [Safety Guardrail] Performing Pre-Flight Moderation Check on Goal ---")
            report = lc_helpers.moderate(goal, self.llm)
            trace.log_moderation("pre", report)

            if not report["available"]:
                print("\n🛑 Moderation could not be performed. Execution halted (fail-safe).")
                print(f"   Reason: {report['error']}")
                trace.finalize("Halted: moderation unavailable")
                return None, trace

            if report["flagged"]:
                print("\n🛑 Goal failed pre-flight moderation. Execution halted.")
                flagged = [k for k, v in report["categories"].items() if v]
                print("   Categories:", ", ".join(flagged) or "unspecified")
                trace.finalize("Halted: pre-flight moderation")
                return None, trace

        result, trace = lc_engine.context_engine(
            goal, self.graph, tracker=self.tracker, trace=trace
        )

        if result and moderation_active:
            text = result if isinstance(result, str) else str(result)
            report = lc_helpers.moderate(text, self.llm)
            trace.log_moderation("post", report)

            if report["flagged"] or not report["available"]:
                reason = ("failed post-flight moderation" if report["flagged"]
                          else "could not be checked (moderation unavailable)")
                print(f"\n🛑 Generated output {reason} and will be redacted.")
                result = ("[Content flagged as potentially harmful by moderation "
                          "policy and has been redacted.]")
                trace.final_output = result
                trace.status = "Redacted: post-flight moderation"

        return result, trace

    # ------------------------------------------------------------------ #
    def plan_only(self, goal: str) -> lc_engine.Plan:
        """Produce the plan without executing it. Useful for teaching and audit.

        Costs exactly one planning call: no retrieval, no generation.
        """
        if self._planner is None:
            self._planner = lc_engine.build_planner(
                self.llm, self.toolkit.get_capabilities_description()
            )
        return self._planner(goal)

    def capabilities(self) -> str:
        return self.toolkit.get_capabilities_description()


def build_context_engine(config: Optional[dict] = None,
                         verbose: bool = True) -> ContextEngine:
    """
    Wire the whole engine together.

        engine = build_context_engine()
        result, trace = engine.run("Summarize the NDA.")

    Any key in lc_helpers.CONFIG can be overridden:

        engine = build_context_engine({"k_knowledge": 5})

    Do NOT override embedding_model unless you have re-ingested the index; the
    stored vectors are 1536-dimensional text-embedding-3-small.
    """
    cfg = dict(lc_helpers.CONFIG)
    if config:
        unknown = set(config) - set(cfg)
        if unknown:
            raise KeyError(
                f"Unknown configuration key(s): {sorted(unknown)}. "
                f"Valid keys: {sorted(cfg)}"
            )
        cfg.update(config)

    llm = lc_helpers.build_llm(cfg)
    base_llm = lc_helpers.base_model(llm)
    embeddings = lc_helpers.build_embeddings(cfg)
    context_store, knowledge_store = lc_helpers.build_stores(embeddings, cfg)
    blueprint_retriever, knowledge_retriever = lc_helpers.build_retrievers(
        context_store, knowledge_store, cfg
    )

    tools = lc_agents.build_agents(
        llm, blueprint_retriever, knowledge_retriever, lc_helpers.sanitize
    )
    toolkit = lc_registry.build_toolkit(tools)
    tracker = lc_helpers.UsageTracker()

    graph = lc_engine.build_engine(
        llm, toolkit, tracker=tracker, count_tokens=lc_helpers.count_tokens
    )

    if verbose:
        print("Context Engine assembled (LangChain Edition).")
        print(f"   index      : {cfg['index_name']}")
        print(f"   generation : {cfg['generation_model']}")
        print(f"   embeddings : {cfg['embedding_model']}")
        print(f"   namespaces : {cfg['namespace_context']} (k={cfg['k_blueprint']}), "
              f"{cfg['namespace_knowledge']} (k={cfg['k_knowledge']})")
        print(f"   agents     : {', '.join(toolkit.names())}")

    return ContextEngine(
        config=cfg, llm=llm, base_llm=base_llm, embeddings=embeddings,
        context_store=context_store, knowledge_store=knowledge_store,
        blueprint_retriever=blueprint_retriever, knowledge_retriever=knowledge_retriever,
        toolkit=toolkit, graph=graph, tracker=tracker, tools=tools,
    )


# =============================================================================
# Route B — the idiomatic create_agent alternative, for comparison
# =============================================================================

def build_react_agent(engine: ContextEngine):
    """
    The same four tools handed to LangChain's standard agent builder.

    Difference from Route A: there is no plan object. The model calls one tool,
    sees the result, and decides what to do next. Six lines instead of sixty —
    at the cost of the up-front, inspectable, auditable plan that the Glass Box
    design depends on. Run both and compare the traces.

    Note also what is lost besides the plan: the per-step token attribution, the
    resolved-context record, and the ability to inspect and approve the work
    before any of it is paid for.
    """
    from langchain.agents import create_agent

    return create_agent(
        model=engine.base_llm,
        tools=engine.tools,
        system_prompt=(
            "You are the strategic core of the Context Engine. Achieve the "
            "user's goal using the available specialists. Typically: retrieve a "
            "Semantic Blueprint with the Librarian, gather grounded facts with "
            "the Researcher, then produce the deliverable with the Writer. "
            "Never invent facts that are not in the retrieved material."
        ),
    )


logging.info("LangChain Context Engine assembly module loaded.")


## Installation and environment setup


In [ ]:
# Install the LangChain stack and load API keys from Colab Secrets.
# The %%writefile cells above only wrote text to disk, so nothing has been
# imported yet and this is the first cell that needs the network.
import lc_utils

installed = lc_utils.install_dependencies()
assert installed, "Installation failed. Read the pip output above before continuing."

# Reads Colab Secrets and exports them as environment variables.
#   API_KEY            -> OPENAI_API_KEY
#   PINECONE_API_KEY   -> PINECONE_API_KEY
#   LANGSMITH_API_KEY  -> LANGSMITH_API_KEY (optional, enables hosted tracing)
#
# Unlike the original, no client objects are created here and none are passed
# around later. LangChain builds its own clients from the environment.
ok = lc_utils.initialize_environment()
assert ok, "Fix the secrets above before continuing."


## Pre-flight: inspect the Pinecone index

Two failure modes cost you a whole run and report nothing: an empty namespace
returns zero documents with no error, and a dimension mismatch returns irrelevant
answers with no error. This one call rules out both for the price of a metadata
request.


In [ ]:
#@title Pre-flight: confirm the index is populated and correctly dimensioned
index_ok = lc_utils.check_index("genai-mas-mcp-ch3")
assert index_ok, "Index pre-flight failed. Run the Chapter 8 and 9 ingestion notebooks first."


# II. Engine assembly

One call builds the model, the embeddings, both vector stores, both retrievers,
the four agents, the toolkit, the token tracker and the compiled LangGraph.


In [ ]:
import lc_boot

engine = lc_boot.build_context_engine()


## Verify the retrieval layer before running anything

This is the single most important check in the whole port. One Pinecone index,
two namespaces, **two different `text_key` values**, two different `k` values.
Get `text_key` wrong and documents come back with empty `page_content` — no
error, just silently useless context.


In [ ]:
#@title Retrieval smoke test (run this before any Control Deck)
# Blueprint side: ContextLibrary, text_key='blueprint_json', k=1
bp_docs = engine.blueprint_retriever.invoke("a formal structured legal summary")
print(f"ContextLibrary -> {len(bp_docs)} document(s)")
for d in bp_docs:
    print("  page_content[:200]:", repr(d.page_content[:200]))
    print("  metadata keys     :", list(d.metadata.keys()))

print()

# Knowledge side: KnowledgeStore, text_key='text', k=3
kn_docs = engine.knowledge_retriever.invoke("confidentiality obligations termination notice")
print(f"KnowledgeStore -> {len(kn_docs)} document(s)")
for d in kn_docs:
    print("  source:", d.metadata.get("source"), "|", repr(d.page_content[:120]))

assert bp_docs and bp_docs[0].page_content, \
    "ContextLibrary returned empty text: check text_key='blueprint_json'"
assert kn_docs and kn_docs[0].page_content, \
    "KnowledgeStore returned empty text: check text_key='text'"
print("\nRetrieval layer OK.")


## The capabilities block — generated, not hand-written

In the original, `registry.get_capabilities_description()` was a 30-line f-string
maintained by hand and kept in sync with four function signatures by discipline
alone. Here it is derived from the tools' own Pydantic schemas, so it cannot
drift out of sync with the code.

The *rendering format* is still ours — LangChain describes a tool to a model, but
not in the `ROLE:`/`INPUTS:` shape this Planner prompt expects. What the framework
removed is the duplication, not the design.


In [ ]:
#@title Inspect the auto-generated capabilities
print(engine.capabilities())


## Trace dashboard

The dashboard is carried over from the original notebook with its CSS unchanged.
Three additions: the execution **plan** has its own card, each step shows the
provider's **real billed tokens** alongside the context-size measurement, and a
**failed step renders in red with its error text** instead of the run ending
silently.

Read the pills as:
- **CTX IN / CTX OUT** — size of the context handed to the step and of what it returned. This is the original notebook's metric and what the Summarizer's *SAVED* figure comes from.
- **LLM IN / LLM OUT** — tokens the provider actually billed for this step, from `usage_metadata`. A retrieval-only step (Librarian) shows *retrieval only* because it makes no model call.

Expect these numbers to differ from the original notebook's screenshots. They
measure different things, and these are exact.


In [ ]:
from lc_dashboard import render_trace_dashboard

print("Dashboard ready.")


## Engine Room

The same `execute_and_display()` entry point as the original, so the Control Decks
below look almost identical. Note the shorter signature: `client`, `pc`,
`index_name`, the model names and the namespaces are all baked into the assembled
`engine`.

One contract change worth knowing: a trace is **always** returned, including when
a goal is blocked by pre-flight moderation. A blocked goal is an audit event, and
it now renders like any other run.


In [ ]:
def execute_and_display(goal, moderation_active=False, ctx_engine=None,
                        label="LangGraph Plan-and-Execute"):
    """Run the LangChain Context Engine and render the HTML dashboard."""
    ctx_engine = ctx_engine or engine

    result, trace = ctx_engine.run(goal, moderation_active=moderation_active)

    render_trace_dashboard(trace, engine_label=label)
    return result


# III. Control Decks

The same goals as the original notebook, so outputs can be compared side by side.

1. Change the `goal` variable.
2. Run the cell.

The configuration dictionary is gone: it lives in `engine` now. To change models
or namespaces, rebuild with
`engine = lc_boot.build_context_engine({"generation_model": "..."})`.


## I — Marketing


In [ ]:
#@title CONTROL DECK: Moderation
# A simple, safe goal that exercises the full moderation workflow:
# pre-flight on the goal, post-flight on the generated output.
goal = "Summarize the key points of the QuantumDrive"

execute_and_display(goal, moderation_active=True)


In [ ]:
#@title Product Marketing Copy Generation
goal = ("Analyze the ChronoTech press release and summarize their core product "
        "messaging and value proposition. Please cite your sources.")

execute_and_display(goal, moderation_active=False)


In [ ]:
#@title Writing a brand pitch recommendation
# Tests the Researcher's ability to report a negative finding and the Writer's
# ability to handle it gracefully, without hallucinating.
goal = "Write a persuasive pitch on our brand tone and voice guide"

execute_and_display(goal, moderation_active=False)


## II — Legal


In [ ]:
#@title CONTROL DECK: Moderation
# The goal is worded to disambiguate "summarize": retrieve first, then summarize.
# A goal of "Summarize the key points of the NDA" leaves the Planner ambiguous
# about whether to use the Researcher or jump straight to the Summarizer.
goal = ("First, retrieve the content of the Non-Disclosure Agreement (NDA) from "
        "the knowledge base. Then, summarize its key points.")

execute_and_display(goal, moderation_active=True)


In [ ]:
#@title CONTROL DECK TEMPLATE 1: High-Fidelity RAG
# Exercises the high-fidelity Researcher: retrieval with `source` metadata,
# per-chunk sanitization, and citation generation.
goal = ("What are the key confidentiality obligations in the Service Agreement v1, "
        "and what is the termination notice period? Please cite your sources.")

# LIMIT TEST — uncomment to watch the sanitizer drop a poisoned chunk while the
# remaining chunks still produce an answer. Look for the
# "[Sanitizer] Potential threat detected" warning in the log output, and note
# that the poisoned document does NOT appear in the Sources list.
# goal = "What did Mr. Smith advise his client regarding the assets?"

execute_and_display(goal, moderation_active=False)


# IV. The Glass Box — inspect a plan before executing it

The engine's central claim is that behaviour comes from context, not from
hard-coded rules. Route A makes that auditable: the plan is a validated object
you can read *before* anything runs and before any money is spent.

Change the goal and watch the plan change while the code stays identical. That is
the domain-agnostic thesis, demonstrated rather than asserted.


In [ ]:
#@title Plan-only: no execution, no cost beyond one planning call
import json

for goal in [
    "First, retrieve the content of the Non-Disclosure Agreement (NDA) from the knowledge base. Then, summarize its key points.",
    "Analyze the ChronoTech press release and summarize their core product messaging and value proposition. Please cite your sources.",
]:
    plan = engine.plan_only(goal)
    print("=" * 100)
    print("GOAL:", goal)
    print("-" * 100)
    for step in plan.plan:
        args = step.input.model_dump(exclude_none=True)
        print(f"  {step.step}. {step.agent:<11} {json.dumps(args, ensure_ascii=False)[:150]}")
    print()


# V. Route B — the same tools, the idiomatic way

Route A (everything above) reproduces the original architecture: **plan first,
then execute**. Route B hands the identical four tools to LangChain's standard
`create_agent` and lets the model decide what to call, one step at a time.

| | Route A — LangGraph | Route B — `create_agent` |
|---|---|---|
| Orchestration code | ~60 lines | ~6 lines |
| Plan artifact | Yes, inspectable before execution | None |
| Auditability | High | Lower — behaviour emerges at run time |
| Per-step token attribution | Yes | No |
| Approval before spend | Possible (`plan_only`) | Not possible |
| Fidelity to the original | Faithful port | A different, simpler system |

Run both on the same goal and compare. The point of this section is the contrast:
Route B is dramatically shorter, and the up-front plan is what you trade away to
get it. Neither is wrong; they belong to different tiers of the delegation
gradient, and the choice should be deliberate.


In [ ]:
#@title Route B: create_agent
import json
from IPython.display import Markdown, display

react_agent = lc_boot.build_react_agent(engine)

goal = ("What are the key confidentiality obligations in the Service Agreement v1, "
        "and what is the termination notice period? Please cite your sources.")

response = react_agent.invoke({"messages": [("user", goal)]})

# Show the tool-calling trajectory the model chose for itself.
print("TRAJECTORY")
print("-" * 80)
for m in response["messages"]:
    kind = m.__class__.__name__
    calls = getattr(m, "tool_calls", None)
    if calls:
        for c in calls:
            print(f"  {kind:<12} -> CALL {c['name']}({json.dumps(c['args'])[:100]})")
    elif kind == "ToolMessage":
        print(f"  {kind:<12} <- {str(m.content)[:100]}...")
    elif m.content:
        print(f"  {kind:<12}    {str(m.content)[:100]}...")

print()
print("=" * 80)
print("FINAL ANSWER")
print("=" * 80)
display(Markdown(response["messages"][-1].content))


# VI. Appendix

## Token accounting for the most recent run

`UsageTracker` is reset at the start of every `engine.run()`, so these figures
describe the last run only, not the session.


In [ ]:
print(engine.tracker.summary())


# VII. What LangChain does not provide

This notebook is a port onto LangChain, not a replacement by LangChain. The
framework supplies the substrate; the following components have no framework
equivalent and are written out in full in the modules above. They are, not
coincidentally, the parts the book is about.

| Component | Where | Why there is no LangChain equivalent |
|---|---|---|
| **Per-chunk sanitization** | `lc_helpers.sanitize`, used inside `lc_agents.Researcher` | Agent middleware operates on a whole message. It cannot drop chunk 2 while keeping chunks 1 and 3, which is exactly the behaviour that lets a poisoned corpus still produce a cited answer. |
| **Moderation** | `lc_helpers.moderate` | `langchain-core` v1 ships no moderation wrapper; the legacy `OpenAIModerationChain` moved to `langchain-classic`. The OpenAI endpoint is called directly, with a documented fail-safe policy that distinguishes *flagged* from *could not be checked*. |
| **Context chaining** | `lc_engine.resolve_dependencies` | LangGraph carries state between nodes. It does not give the model a dataflow language in which to *declare* dependencies at plan time — `$$STEP_N_OUTPUT$$` is this engine's own. |
| **Plan-and-Execute** | `lc_engine.build_engine` | Removed from LangChain core; what remains in `langchain-experimental` is unmaintained. The two-node graph is built by hand on the Graph API. |
| **Per-step token attribution** | `lc_helpers.UsageTracker` | LangChain reports usage per model call. Attributing it to a *plan step* requires snapshotting against the execution cursor. |
| **The audit trace and dashboard** | `lc_engine.LangChainTrace`, `lc_dashboard` | LangSmith is a hosted run viewer. It does not record `planned_input` vs `resolved_context` vs `tokens_saved` per plan step, and it is not an offline, embeddable artifact. |
| **Capability rendering** | `lc_registry.get_capabilities_description` | The *schema* is generated by LangChain; the `ROLE:`/`INPUTS:` block the Planner prompt reads is not. |

Two shims are also worth naming honestly, because they exist to work *around* the
framework rather than to use it: `lc_helpers.base_model()` unwraps `RunnableRetry`
because `.with_retry()` does not forward `with_structured_output()` or
`get_num_tokens()`; and `lc_helpers._openai_client()` resolves a moderation client
through three tiers, ending in a directly constructed `openai.OpenAI()` so that a
change in `langchain-openai`'s internals degrades gracefully instead of disabling
the guardrail.

## LangSmith

If you added a `LANGSMITH_API_KEY` secret, every run above was recorded
automatically — full inputs, outputs, timings and token counts for every node,
chain and tool, with no instrumentation code. Open <https://smith.langchain.com>
and look for the project `universal-context-engine-langchain`.

This replaces the *observability* half of the original `ExecutionTrace`, not the
*audit* half. `LangChainTrace` is kept regardless, so the notebook still works and
still renders its dashboard with no LangSmith account and no network egress to a
third party — which is usually the deciding factor in a regulated deployment.


## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| Retrieval returns 0 documents | Wrong or missing namespace | Namespaces are case-sensitive: `ContextLibrary`, `KnowledgeStore` |
| Documents have empty `page_content` | Wrong `text_key` | `blueprint_json` for ContextLibrary, `text` for KnowledgeStore. The Librarian and Researcher now log an explicit error when this happens |
| Answers are irrelevant but not empty | Embedding model mismatch | Must be `text-embedding-3-small`. `check_index()` verifies the 1536 dimension |
| `KeyError` on a prompt fragment | Unescaped `{` in a prompt | Double every literal brace: `{{` and `}}` |
| Planner picks the wrong agent | Vague tool docstring | The docstring *is* the interface the model reads. Rewrite it |
| `Agent 'X' not found in registry` | Shouldn't happen | The `Literal` in `PlanStep` prevents invented names, and `build_engine` asserts the toolkit matches it |
| Moderation blocks everything | The endpoint could not be reached | Fail-safe is deliberate. The console prints the underlying error and the trace status reads *Halted: moderation unavailable*, distinct from *Halted: pre-flight moderation* |
| `ImportError` after editing a `%%writefile` cell | Stale import cache | Restart the runtime after re-running a module cell |
| Slow first run | Cold Pinecone connection | Normal. Subsequent runs are faster |

## Modifying the engine

- **Add an agent:** write a new `@tool` in `lc_agents.py`, return it from `build_agents`, add its name to `lc_engine.AGENT_NAMES` **and** to the `Literal` in `PlanStep`, and add any new argument to `AgentInput`. `build_engine()` raises at assembly time if you forget one of these. The capabilities block updates itself.
- **Change retrieval depth:** `lc_boot.build_context_engine({"k_knowledge": 5})`. Unknown keys raise immediately rather than being silently ignored.
- **Change the model:** `lc_boot.build_context_engine({"generation_model": "..."})`. Do **not** change `embedding_model` unless you re-ingest — the stored vectors are 1536-dimensional.
- **Resume failed runs:** pass a checkpointer to `graph.compile()` in `lc_engine.build_engine`. LangGraph will then restart at the failing step rather than from the beginning.
- **Approve plans before execution:** call `engine.plan_only(goal)` first, or add LangGraph's `interrupt()` between the `plan` and `execute` nodes for a true human-in-the-loop gate.
